<a href="https://colab.research.google.com/github/SRET-College/Sem-6-NN-and-DL/blob/main/NN_and_DL_5b_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Madaline Implementation

**Madaline (Many Adalines)** is a type of artificial neural network introduced by Bernard Widrow and Ted Hoff in 1960. It's an extension of the Adaline (Adaptive Linear Neuron) model, designed to solve problems that are not linearly separable.

### Key Components:
1.  **Adaline Units (Hidden Layer)**: Madaline consists of several Adaline units in its hidden layer. Each Adaline unit is a single-layer neural network that computes a weighted sum of its inputs and applies a hard-limit activation function (typically a sign function).
2.  **Output Layer (Fixed Logic)**: The outputs of the Adaline units in the hidden layer are fed into a fixed logic gate (e.g., an OR gate or an AND gate) to produce the final output. The weights of this output layer are not adjusted during training.
3.  **Training Rule (MR-I Learning Rule)**: Madaline uses a unique training algorithm called the MR-I (Madaline Rule I). This rule identifies the Adaline units in the hidden layer that are "least committed" (i.e., whose net input is closest to zero) and adjusts their weights if the overall network's output is incorrect. The idea is to change the minimum number of weights to correct the error.

### How it works:
-   For a given input, each Adaline in the hidden layer calculates its output.
-   These outputs are then combined by the fixed logic gate (e.g., if any hidden Adaline outputs +1, the Madaline outputs +1 for an OR gate).
-   If the Madaline's output is incorrect, the MR-I rule identifies the hidden Adaline unit(s) whose weights should be adjusted. It typically changes the weights of the hidden Adaline unit(s) that are closest to changing their output, to correct the overall error.

Below is a basic implementation of Madaline. We'll start by implementing the `Adaline` class, which serves as the building block for the Madaline network, and then the `Madaline` class itself. We'll demonstrate it with a simple XOR-like problem, keeping in mind that the standard Madaline with MR-I is primarily for binary classification and can struggle with complex non-linear problems without careful design or modifications.

In [ ]:
import numpy as np

class Adaline:
    def __init__(self, input_size, learning_rate=0.01):
        self.weights = np.random.rand(input_size + 1) * 0.1 - 0.05 # Initialize small random weights
        self.learning_rate = learning_rate

    def predict(self, inputs):
        # Add bias input (1) to the inputs
        summation = np.dot(inputs, self.weights[1:]) + self.weights[0]
        return np.where(summation >= 0, 1, -1) # Hard-limit activation

    def train(self, inputs, target_output):
        # Add bias input (1) to the inputs for weight adjustment
        net_input = np.dot(inputs, self.weights[1:]) + self.weights[0]
        output = np.where(net_input >= 0, 1, -1)

        error = target_output - net_input # Adaline uses linear output for weight update

        # Update weights
        self.weights[1:] += self.learning_rate * error * inputs
        self.weights[0] += self.learning_rate * error

        return output # Return the actual output after training

In [ ]:
class Madaline:
    def __init__(self, input_size, num_adalines, learning_rate=0.01):
        self.adalines = [Adaline(input_size, learning_rate) for _ in range(num_adalines)]
        self.num_adalines = num_adalines
        # For simplicity, we'll use a fixed OR logic for the output layer
        # i.e., if any Adaline outputs 1, Madaline outputs 1.

    def predict(self, inputs):
        adaline_outputs = np.array([adaline.predict(inputs) for adaline in self.adalines])
        # Fixed OR logic: if any adaline output is 1, final output is 1, else -1
        return 1 if np.any(adaline_outputs == 1) else -1

    def train(self, X, y, epochs=100):
        for epoch in range(epochs):
            total_error = 0
            for inputs, target_output in zip(X, y):
                adaline_net_inputs = []
                adaline_outputs = []
                for adaline in self.adalines:
                    net_input = np.dot(inputs, adaline.weights[1:]) + adaline.weights[0]
                    adaline_net_inputs.append(net_input)
                    adaline_outputs.append(np.where(net_input >= 0, 1, -1))

                adaline_outputs = np.array(adaline_outputs)
                madaline_output = 1 if np.any(adaline_outputs == 1) else -1

                if madaline_output != target_output:
                    # Madaline Rule I (MR-I)
                    # Find the Adaline unit whose output needs to be flipped with minimum weight change
                    # This often means finding the Adaline with the smallest absolute net input
                    # or closest to zero output.

                    # For a +1 target, we need at least one +1 hidden output.
                    # If current output is -1 (meaning all hidden are -1), flip the 'least committed' one to +1.
                    if target_output == 1:
                        # Find the Adaline unit that outputted -1 and is closest to 0 (to flip to +1)
                        candidate_adalines_idx = [i for i, out in enumerate(adaline_outputs) if out == -1]
                        if candidate_adalines_idx:
                            min_net_input_idx = candidate_adalines_idx[np.argmin([np.abs(adaline_net_inputs[i]) for i in candidate_adalines_idx])]
                            self.adalines[min_net_input_idx].train(inputs, 1) # Force this Adaline to output 1

                    # For a -1 target, we need all -1 hidden outputs.
                    # If current output is +1 (meaning at least one hidden is +1), flip all +1 outputs to -1.
                    elif target_output == -1:
                        for i, out in enumerate(adaline_outputs):
                            if out == 1:
                                self.adalines[i].train(inputs, -1) # Force this Adaline to output -1
                    total_error += 1
            if total_error == 0:
                print(f"Converged in {epoch+1} epochs.")
                break
        print(f"Training finished after {epoch+1} epochs. Total errors in last epoch: {total_error}")

In [ ]:
# Example Usage: XOR-like problem
# Input: [[0,0], [0,1], [1,0], [1,1]]
# Target: [-1, 1, 1, -1] (using -1 for 0, 1 for 1)

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([-1, 1, 1, -1]) # XOR targets

# Instantiate Madaline with 2 input features and 2 Adaline units in the hidden layer
madaline = Madaline(input_size=2, num_adalines=2, learning_rate=0.01)

print("Initial predictions:")
for inputs, target in zip(X, y):
    prediction = madaline.predict(inputs)
    print(f"Input: {inputs}, Expected: {target}, Predicted: {prediction}")

print("\nTraining Madaline...")
madaline.train(X, y, epochs=1000)

print("\nPredictions after training:")
for inputs, target in zip(X, y):
    prediction = madaline.predict(inputs)
    print(f"Input: {inputs}, Expected: {target}, Predicted: {prediction}")


Initial predictions:
Input: [0 0], Expected: -1, Predicted: 1
Input: [0 1], Expected: 1, Predicted: 1
Input: [1 0], Expected: 1, Predicted: 1
Input: [1 1], Expected: -1, Predicted: 1

Training Madaline...
Converged in 6 epochs.
Training finished after 6 epochs. Total errors in last epoch: 0

Predictions after training:
Input: [0 0], Expected: -1, Predicted: -1
Input: [0 1], Expected: 1, Predicted: 1
Input: [1 0], Expected: 1, Predicted: 1
Input: [1 1], Expected: -1, Predicted: -1
